# 03 Reflection loop：GPT 看圖檢查 → 不合格就重生

對指定方案（`SCHEME`）的每一格：GPT（gpt-4.1）對照分鏡檢查 → 依規則決定 通過／改寫 prompt／換 seed → 重生後再檢查，**每格最多重生 3 次**，保留最好的一張。

- 改寫 prompt：用評審改寫的 image_prompt 重新組合（外觀仍只定義一次），seed 不變
- 換 seed：seed + 1000 × 嘗試次數
- 一律對照**原本的分鏡**檢查（改寫的 prompt 只是生圖手段）
- 結果存成新方案 `<SCHEME>_R`（每格的嘗試紀錄在 json 的 `reflection` 欄位，嘗試的圖在 `<SCHEME>_R/try1..3/`），可與 `<SCHEME>` 比較，即企畫書的 **reflection 消融實驗**
- 費用：每次檢查約 0.009 美金（30 格初檢約 0.26 美金，加上重生後的複檢）；檢查結果快取在 Drive，斷線重跑不重複付費

**執行前**：T4 GPU；左側「鑰匙」Secrets 新增 `OPENAI_API_KEY`（OpenAI 的 key，`sk-` 開頭），開啟 Notebook access。**不要**把 OpenAI key 填到 `GH_TOKEN`。

In [ ]:
# ==== 設定 ====
REPO_URL = "https://github.com/77JoyWang77/1151-AI-Final.git"
REPO_DIR = "/content/1151-AI-Final"
DRIVE_ROOT = "/content/drive/MyDrive/02 課程/1151-AI-Final"  # 生成結果存這裡
SCHEME = "B_s0.3"          # 要做 reflection 的方案（需已由 01／02 生成）
STORIES = [1, 2, 3, 4, 5]
MAX_TRIES = 3
CPU_OFFLOAD = False        # 若出現 CUDA out of memory 改成 True（較慢）

from google.colab import drive
drive.mount("/content/drive")

### 取得程式碼

repo 目前是 public，不用設定。若改成 **private**：左側欄「鑰匙」圖示（Secrets）新增 `GH_TOKEN`，值為 GitHub personal access token（只需 repo 讀取權限），並開啟「Notebook access」。

若重啟過執行階段（`/content` 會被清空），要從第一格重新執行。

In [ ]:
import base64
import os
import subprocess


def clone(*git_opts):
    return subprocess.run(["git", *git_opts, "clone", "-q", REPO_URL, REPO_DIR], capture_output=True, text=True)


if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} pull -q --ff-only
else:
    !rm -rf {REPO_DIR}  # 清掉上次 clone 失敗留下的殘骸
    result = clone()  # public repo 不需要 token
    if result.returncode != 0:
        try:
            from google.colab import userdata
            token = (userdata.get("GH_TOKEN") or "").strip()
        except Exception:
            token = ""
        # 不印出 token 內容；檢查前綴，避免誤填成 OpenAI key 等其他密鑰
        if not token.startswith(("ghp_", "github_pat_")):
            raise RuntimeError("clone 失敗，且 Secrets 沒有有效的 GH_TOKEN（GitHub token 以 ghp_ 或 github_pat_ 開頭）\n"
                               + result.stderr)
        # token 放在 HTTP header 而不是網址，git 的錯誤訊息不會把它印出來
        auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        result = clone("-c", f"http.extraHeader=Authorization: Basic {auth}")
        if result.returncode != 0:
            raise RuntimeError("用 GH_TOKEN clone 仍失敗：\n" + result.stderr)
assert os.path.isdir(f"{REPO_DIR}/src/comic_agent"), "clone 失敗：src/comic_agent 不存在"
!git -C {REPO_DIR} log --oneline -1

In [ ]:
!pip install -q diffusers transformers accelerate python-dotenv openai

### OpenAI key

從 Colab Secrets 讀取，不會印出內容。GPT 檢查結果快取在 Drive（`cache/llm/`），必須在 import `comic_agent` **之前**設定。

In [ ]:
import os

from google.colab import userdata

key = (userdata.get("OPENAI_API_KEY") or "").strip()
assert key.startswith("sk-"), "請在 Colab Secrets 新增 OPENAI_API_KEY（OpenAI 的 key，以 sk- 開頭），並開啟 Notebook access"
os.environ["OPENAI_API_KEY"] = key
os.environ["LLM_CACHE_DIR"] = f"{DRIVE_ROOT}/cache/llm"  # GPT 檢查結果存 Drive：斷線重跑不重複付費
del key
print("OpenAI key 已設定")

In [ ]:
import importlib
import json
import subprocess
import sys
import time

if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
importlib.invalidate_caches()  # 同一個 kernel 若在 clone 前 import 失敗過，Python 會記住「路徑不存在」
from comic_agent import render

CONFIG = render.GenConfig()
OUT_ROOT = f"{DRIVE_ROOT}/outputs/render"
GIT_COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], text=True).strip()

boards = render.load_frozen_storyboards(f"{REPO_DIR}/data/storyboards")
from comic_agent import llm, reflect

REVIEW_OUT = f"{SCHEME}_R"
jobs = [j for j in render.iter_jobs(boards, SCHEME) if j["story_no"] in STORIES]
missing = [j for j in jobs if not render.panel_paths(OUT_ROOT, j)[0].exists()]
assert not missing, f"{SCHEME} 還有 {len(missing)} 格沒有生成，請先跑 01／02"
print(f"{SCHEME}：{len(jobs)} 格 → 輸出 {REVIEW_OUT}；評審模型 {reflect.REVIEW_MODEL}；commit {GIT_COMMIT}")

### 載入 SDXL

第一次執行會下載約 7 GB 權重，需要幾分鐘。VAE 換成 `sdxl-vae-fp16-fix`，原版 VAE 在 fp16 下會溢位產生黑圖。

In [ ]:
import torch
from diffusers import AutoencoderKL, DPMSolverMultistepScheduler, StableDiffusionXLPipeline

vae = AutoencoderKL.from_pretrained(CONFIG.vae_id, dtype=torch.float16)
pipe = StableDiffusionXLPipeline.from_pretrained(
    CONFIG.model_id, vae=vae, dtype=torch.float16, variant="fp16", use_safetensors=True,
)
# DPM++ 2M Karras
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, algorithm_type="dpmsolver++", solver_order=2, use_karras_sigmas=True,
)
if CPU_OFFLOAD:
    pipe.enable_model_cpu_offload()
else:
    pipe.to("cuda")
pipe.set_progress_bar_config(disable=True)
print(torch.cuda.get_device_name(0), f"{torch.cuda.memory_allocated() / 2**30:.1f} GB")

In [ ]:
from pathlib import Path

from PIL import Image, ImageDraw


def generate(job, out_root=OUT_ROOT, skip_existing=True, pipe_kwargs=None, record=None):
    """生成一格並存圖與紀錄；已存在且設定相同就略過（斷線續跑）。

    「設定」= prompt 加上 record 的欄位（例如 IP-Adapter scale、參考圖）。
    任一項改變（例如換了畫風或參考圖）都會重新生成並覆蓋，避免沿用舊圖。
    """
    record = record or {}
    png, js = render.panel_paths(out_root, job)
    if skip_existing and png.exists() and js.exists():
        old = json.loads(js.read_text(encoding="utf-8"))
        if all(old.get(k) == v for k, v in {"prompt": job["prompt"], **record}.items()):
            return png, False
    generator = torch.Generator("cpu").manual_seed(job["seed"])  # CPU generator：跨機器較可重現
    t0 = time.time()
    image = pipe(
        prompt=job["prompt"],
        negative_prompt=job["negative_prompt"],
        num_inference_steps=CONFIG.steps,
        guidance_scale=CONFIG.cfg,
        width=CONFIG.width,
        height=CONFIG.height,
        generator=generator,
        **(pipe_kwargs or {}),
    ).images[0]
    seconds = round(time.time() - t0, 1)
    png.parent.mkdir(parents=True, exist_ok=True)
    image.save(png)
    render.write_record(js, job, CONFIG, seconds=seconds, git_commit=GIT_COMMIT,
                        gpu=torch.cuda.get_device_name(0), **record)
    return png, True


def contact_sheet(pngs, cols=3, thumb=384, labels=None):
    """把多張圖排成總覽圖；labels 為每一列的標題。缺圖的位置留白。"""
    rows = (len(pngs) + cols - 1) // cols
    head = 28 if labels else 0
    sheet = Image.new("RGB", (thumb * cols, (thumb + head) * rows), "white")
    draw = ImageDraw.Draw(sheet)
    for i, p in enumerate(pngs):
        r, c = divmod(i, cols)
        if p is not None and Path(p).exists():
            sheet.paste(Image.open(p).resize((thumb, thumb)), (c * thumb, r * (thumb + head) + head))
    for r, text in enumerate(labels or []):
        draw.text((6, r * (thumb + head) + 6), text, fill="black")
    return sheet

### 方案 B：載入 IP-Adapter

scale 由方案名稱讀出（`B_s0.3` → 0.3），參考圖由該方案第 1 格的紀錄讀出，與原本生成時完全相同。方案 A 會略過這一格。

In [ ]:
IP_WEIGHT = "ip-adapter-plus_sdxl_vit-h.safetensors"


def pipe_kwargs_for(job):
    return {}


def record_for(job):
    return {}


if SCHEME.startswith("B_s"):
    IP_SCALE = float(SCHEME.split("_s")[1])
    refs = {}
    for s in STORIES:
        rec_path = render.panel_paths(OUT_ROOT, {"scheme": SCHEME, "story_no": s, "panel_index": 1})[1]
        refs[s] = json.loads(rec_path.read_text(encoding="utf-8"))["ref_image"]
    pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name=IP_WEIGHT,
                         image_encoder_folder="models/image_encoder")
    pipe.set_ip_adapter_scale(IP_SCALE)

    def pipe_kwargs_for(job):
        return {"ip_adapter_image": Image.open(Path(OUT_ROOT) / refs[job["story_no"]]).convert("RGB")}

    def record_for(job):
        return {"ip_adapter": IP_WEIGHT, "ip_scale": IP_SCALE, "ref_image": refs[job["story_no"]]}

    print(f"IP-Adapter scale {IP_SCALE}，參考圖：{refs}")

### 逐格 reflection loop

已完成的格（`<SCHEME>_R` 有紀錄）會略過，可斷線續跑。

In [ ]:
import shutil

results = []
for job in jobs:
    board = boards[job["story_no"]]
    panel = board["panels"][job["panel_index"] - 1]
    final_png, final_js = render.panel_paths(OUT_ROOT, {**job, "scheme": REVIEW_OUT})
    if final_js.exists():
        results.append(json.loads(final_js.read_text(encoding="utf-8"))["reflection"])
        continue

    def gen(j, k):
        png, _ = generate({**j, "scheme": f"{REVIEW_OUT}/try{k}"},
                          pipe_kwargs=pipe_kwargs_for(j), record=record_for(j))
        return png

    out = reflect.reflection_loop(job, board, render.panel_paths(OUT_ROOT, job)[0], gen,
                                  review=lambda png: reflect.review_panel(png, panel, board),
                                  max_tries=MAX_TRIES)
    chosen = out["attempts"][out["chosen"]]
    final_png.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(chosen["path"], final_png)
    render.write_record(final_js, {**job, "scheme": REVIEW_OUT, "prompt": chosen["prompt"], "seed": chosen["seed"]},
                        CONFIG, git_commit=GIT_COMMIT, reflection=out, **record_for(job))
    results.append(out)
    trail = " → ".join(a["action"] for a in out["attempts"])
    print(f"story {job['story_no']} panel {job['panel_index']}：{trail}（選第 {out['chosen']} 次）")

### 統計與比較

- 初檢通過率 vs 最終通過率：reflection 救回了多少格
- 重生次數與 API 費用：值不值得（企畫書的消融實驗）
- 並排圖：每篇上排是原方案、下排是 reflection 後

In [ ]:
first_pass = sum(r["attempts"][0]["action"] == "pass" for r in results)
final_pass = sum(r["attempts"][r["chosen"]]["action"] == "pass" for r in results)
regens = sum(len(r["attempts"]) - 1 for r in results)
u = llm.USAGE
cost = u["input_tokens"] / 1e6 * 2.0 + u["output_tokens"] / 1e6 * 8.0  # gpt-4.1 單價（估算）
print(f"初檢通過 {first_pass}/{len(results)} → 最終通過 {final_pass}/{len(results)}，共重生 {regens} 次")
print(f"本次 API：{u['calls']} 次（快取命中不算），估計 {cost:.3f} 美金")

for story_no in STORIES:
    n = len(boards[story_no]["panels"])
    pngs = [render.panel_paths(OUT_ROOT, {"scheme": s, "story_no": story_no, "panel_index": i})[0]
            for s in (SCHEME, REVIEW_OUT) for i in range(1, n + 1)]
    sheet = contact_sheet(pngs, cols=n, thumb=256, labels=[SCHEME, REVIEW_OUT])
    sheet.save(Path(OUT_ROOT) / f"reflect_story_{story_no}.png")
    print(f"story {story_no}《{boards[story_no]['title']}》")
    display(sheet)